# RAD

**Paper**: [Reward-Augmented Decoding: Efficient Controlled Text Generation With a Unidirectional Reward Model](https://arxiv.org/abs/2310.09520)

**Authors**: Haikang Deng, Colin Raffel

RAD (reward-augmented decoding) is an output steering method that performs controlled text generation with a reward model. At each decoding step, RAD scores the top-`top_k` candidate tokens with an auxiliary reward model and shifts their logits by `beta * reward`. The reward model can be any Hugging Face sequence-classification model, and when it is decoder-only and shares the base model's vocabulary RAD caches the reward model's prefix activations across steps (the efficient unidirectional path from the paper).

In this demo, we use a reward model to steer a base language model toward higher-reward continuations on adversarial prompts.

## Method parameters

| parameter             | type    | description                                                                                     |
| --------------------- | ------- | ----------------------------------------------------------------------------------------------- |
| `reward_model_id`     | `str`   | HF model id or local path for an `AutoModelForSequenceClassification` reward model.             |
| `beta`                | `float` | Steering intensity (Algorithm 1's beta). Non-negative; direction is set by `invert`.            |
| `top_k`               | `int`   | Number of candidate tokens scored per step (Algorithm 1's k).                                   |
| `invert`              | `bool`  | Use `1 - reward` as the shift (steer away from the scored attribute).                           |
| `score_index`         | `int`   | Output column of the reward model read as the score.                                            |
| `score_transform`     | `str`   | Map head outputs to [0, 1] before selecting `score_index`: `"none"`, `"sigmoid"`, or `"softmax"`. |
| `reward_model_kwargs` | `dict`  | Extra kwargs for `AutoModelForSequenceClassification.from_pretrained()`.                         |
| `include_in_scoring`  | `bool`  | Apply the processor during `compute_logprobs` (one aux forward per reference position).          |
| `efficient`           | `bool`  | Cache reward-model prefix activations across steps when the preconditions hold.                 |


## Setup

If running this from a Google Colab notebook, please uncomment the following cell to install the toolkit. The following block is not necessary if running this notebook from a virtual environment where the package has already been installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The base model used below is gated on Hugging Face, so we log in with a token stored in the `.env` file (after being granted access on the model's Hub page). Uncomment the following if you need to authenticate:

In [2]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: reward-guided continuation

In [3]:
from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.rad.control import RAD
from aisteer360.utils.verbosity import quiet_third_party

quiet_third_party()  # reduce progress bars and info logs

MODEL_NAME = "meta-llama/Llama-3.2-1B"
REWARD_MODEL_ID = "Skywork/Skywork-Reward-V2-Llama-3.2-1B"

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We steer a Llama-3.2-1B base model with a same-family reward model, `Skywork/Skywork-Reward-V2-Llama-3.2-1B`. This reward model is a decoder-only `LlamaForSequenceClassification` whose single output is a scalar preference reward (higher is better), and it shares the Llama-3.2 tokenizer with the base model. Because the reward model is decoder-only and shares the base vocabulary, RAD caches its prefix activations across decoding steps (the paper's efficient unidirectional path), which we leave on via the default `efficient=True`.

The reward head is a Bradley-Terry preference model: its single output column (`score_index=0`) is an unbounded preference score rather than a bounded reward. RAD's processor clamps each candidate value to `[0, 1]` before shifting the logits, so we first map the score into that range with `score_transform="sigmoid"`, which is order-preserving and matches the range the clamp assumes; `invert=False` keeps the shift in favor of higher-reward continuations. Choosing a same-family reward model is the paper's appendix recommendation, and it is what lets RAD feed the base model's own token ids to the reward model without a text round-trip.

`beta` is the steering strength. Since the transformed reward lies in `[0, 1]`, `beta` bounds the maximum per-candidate logit shift; we use `beta=10`. `top_k=20` is the paper's candidate count.

One caveat is worth keeping in mind when reading the outputs. The reward model is trained on chat-templated complete conversations, so its scores on raw partial continuations of a base model are out of its training distribution, and this demo is qualitative. A reward model trained on partial sequences for the target attribute, as in the RAD paper, is the faithful configuration.

In [4]:
rad = RAD(
    reward_model_id=REWARD_MODEL_ID,
    beta=10,
    top_k=20,
    score_index=0,
    score_transform="sigmoid",
    invert=False,
)

We create and steer the `SteeringPipeline` with the above `rad` control. The `steer()` call loads the reward model and, on this decoder-only shared-vocabulary pair, builds the cached reward value.

In [5]:
rad_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[rad],
    device="cuda",
    hf_model_kwargs={"low_cpu_mem_usage": True},
)
rad_pipeline.steer()

#### Controlled text generation via RAD steering

The generation prompt is selected (in the style of the RealToxicityPrompts dataset) in an attempt to induce an undesirable continuation which the reward model steers away from.

In [6]:
tokenizer = rad_pipeline.tokenizer

PROMPT = "I had someone throw me to the ground and say: "

inputs = tokenizer(PROMPT, return_tensors="pt")

In [7]:
gen_params = {
    "max_new_tokens": 100,
    "do_sample": True,
}

steered_output_ids = rad_pipeline.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    runtime_kwargs={},
    **gen_params,
)

print(f"\nSteered response (RAD, beta={rad.beta}):\n")
print(tokenizer.decode(steered_output_ids[0], skip_special_tokens=True))


Steered response (RAD, beta=10):

 “I told you so! You’ve had it too easy! You were the one who got out of here!”  Well he just proved to me once again he really doesn’t get it.  Folks were really disappointed and they were not pleased.  Folks were really really disappointed.  Folks were really really disappointed.  Folks were really really disappointed.  Folks were really really disappointed.  Folks were really really disappointed.  F


#### Comparison against the unsteered model

To see the effect of RAD, we compare against an unsteered pipeline (`controls=[]`) on the same model and generation parameters. The unsteered pipeline does not load the reward model.

In [8]:
baseline_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[],
    device="cuda",
    hf_model_kwargs={"low_cpu_mem_usage": True},
)
baseline_pipeline.steer()

baseline_output_ids = baseline_pipeline.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    runtime_kwargs={},
    **gen_params,
)

print("\nUnsteered response:\n")
print(tokenizer.decode(baseline_output_ids[0], skip_special_tokens=True))


Unsteered response:

 “Do you know how many people are on the streets in America?  There are more than 1 million homeless people in America.  Do you know how many people are on the streets in America?  There are more than 1 million homeless people in America.”  And I said, “Yes, I know.”  And she said, “And do you know how many people are on the streets in America?  There are more than 1 million


#### Composing with sampling parameters

The parameters above are the paper's algorithm parameters. RAD also composes with the usual sampling knobs, which apply around the reward shift rather than in place of it. Note that temperature rescales the effective `beta` (the shift is applied to the logits before temperature scaling), so a run with `temperature` and `top_p` set is a variant of, not a reproduction of, the paper's configuration.

In [9]:
sampling_params = {
    "max_new_tokens": 20,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
}

composed_output_ids = rad_pipeline.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    runtime_kwargs={},
    **sampling_params,
)

print(f"\nSteered response (RAD, beta={rad.beta}, with sampling params):\n")
print(tokenizer.decode(composed_output_ids[0], skip_special_tokens=True))


Steered response (RAD, beta=10, with sampling params):

 “I hope you enjoy getting run over!  Haha ha ha ha ha ha ha ha ha
